# ECMM422J Coursework 1 — ECG Heartbeat Classification with a 1D CNN

**Author:**  Arush Kumar Vishwakarma
**Student ID:** 

## Overview
This notebook builds a 1D Convolutional Neural Network to classify ECG heartbeats from the MIT-BIH Arrhythmia dataset into five categories: Normal, Supraventricular, Ventricular, Fusion, and Unknown/Paced.

The pipeline runs in 9 stages:
1. Setup & reproducibility
2. Load data
3. Preprocessing
4. Visualisation
5. Model architecture
6. Class weight calculation
7. K-fold cross-validation + hyperparameter search
8. Final training
9. Evaluation

---
## Stage 1 — Setup and Reproducibility

Import libraries and lock random seeds so the experiment produces identical results every time.

In [1]:
# Stage 1: Setup and Reproducibility

# Standard library
import os
import random
import json
import time

# Numerical and data libraries
import numpy as np
import pandas as pd

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning library
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical, plot_model

# Classical ML utilities from scikit-learn
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

# ---- Reproducibility: lock random seeds ----
# Without this, every training run gives slightly different numbers
# because weight initialisation, data shuffling, and dropout are random.
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Plot styling ----
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

# ---- Confirm environment ----
print('TensorFlow version:', tf.__version__)
print('Devices available:', tf.config.list_physical_devices())
print('Seed locked to:', SEED)

TensorFlow version: 2.21.0
Devices available: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Seed locked to: 42


---
## Stage 2 — Load the Data

The training and test sets are provided as separate CSV files. Each row is one heartbeat:
the first 187 columns are time-domain ECG samples (already normalised to ~[0,1] by the
dataset authors), and the last column is the integer class label in {0, 1, 2, 3, 4}.

No header row, so we pass `header=None` to pandas.

In [2]:
# Stage 2: Load the training and test data

# File paths - data/ folder sits next to the notebook
TRAIN_PATH = 'data/mitbih_train.csv'
TEST_PATH  = 'data/mitbih_test.csv'

# Read both CSVs. header=None because the files have no column names.
t0 = time.time()
train_df = pd.read_csv(TRAIN_PATH, header=None)
test_df  = pd.read_csv(TEST_PATH,  header=None)
print(f'Loaded both CSVs in {time.time() - t0:.1f}s')

# Split each DataFrame into features (X) and label (y).
# .iloc[:, :-1] = "all rows, all columns except the last" = the 187 features
# .iloc[:, -1]  = "all rows, the last column only"        = the class label
# .values gives us numpy arrays (faster than DataFrame for ML).
# astype enforces dtypes: float32 for features (TF prefers this), int64 for labels.
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = train_df.iloc[:, -1].values.astype(np.int64)

X_test = test_df.iloc[:, :-1].values.astype(np.float32)
y_test = test_df.iloc[:, -1].values.astype(np.int64)

# Sanity prints - shapes, label range, value range
print(f'\nShapes:')
print(f'  X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'  X_test : {X_test.shape}   y_test : {y_test.shape}')

print(f'\nLabel values present (train): {np.unique(y_train)}')
print(f'Label values present (test) : {np.unique(y_test)}')

print(f'\nFeature value range:')
print(f'  Train: [{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'  Test : [{X_test.min():.3f}, {X_test.max():.3f}]')

Loaded both CSVs in 2.6s

Shapes:
  X_train: (87554, 187)   y_train: (87554,)
  X_test : (21892, 187)   y_test : (21892,)

Label values present (train): [0 1 2 3 4]
Label values present (test) : [0 1 2 3 4]

Feature value range:
  Train: [0.000, 1.000]
  Test : [0.000, 1.000]
